In [ ]:
import os
import gc
import torch
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

# 1. Önceki modeli VRAM'den temizle
if "model" in globals():
    del model
    del processor
    gc.collect()
    torch.cuda.empty_cache()

# 2. 448 Modelini doğrudan GPU'ya sabitle
model_id = "google/paligemma2-3b-mix-448"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("448 Modeli GPU'ya yükleniyor...")
processor = AutoProcessor.from_pretrained(model_id)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"  # CPU yavaşlığını ve meta-device uyarısını engeller
)
print("448 Modeli hazır!\n")

# 3. Test fonksiyonu
def test_448(gorsel_yolu, prompt="ocr"):
    if not os.path.exists(gorsel_yolu):
        return "Dosya bulunamadı."

    resim = Image.open(gorsel_yolu).convert("RGB")
    girdi_metni = f"<image>{prompt.strip()}\n"
    inputs = processor(text=girdi_metni, images=resim, return_tensors="pt").to(device)

    with torch.no_grad():
        cikis = model.generate(
            **inputs,
            max_new_tokens=250,        # Satırları ve alt toplamı eksiksiz dökmesi için
            repetition_penalty=1.2,
            do_sample=False
        )

    sonuc = processor.decode(cikis[0], skip_special_tokens=True)
    if prompt in sonuc:
        sonuc = sonuc.split(prompt)[-1].strip()
    return sonuc.strip()

# 4. Karşılaştırma yapılacak fişler
test_dosyalari = ["images-2.jpeg", "images-3.jpeg"]

print("--- 448 MODELİ ÇIKTI TESTİ ---\n")
for dosya in test_dosyalari:
    if os.path.exists(dosya):
        print("=" * 45)
        print(f"📄 Dosya: {dosya}")

        # 1. OCR (Netleşen satırlar)
        print("[OCR Sonucu]:")
        ocr_cikti = test_448(dosya, prompt="ocr")
        print(ocr_cikti)

        # 2. Toplam Tutar
        print("\n[Toplam Tutar Sorusu]:")
        tutar_cikti = test_448(dosya, prompt="answer en What is the total price or amount?")
        print(f"Modelin Doğru Çıktısı: {tutar_cikti}\n")


448 Modeli GPU'ya yükleniyor...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

448 Modeli hazır!

--- 448 MODELİ ÇIKTI TESTİ ---

📄 Dosya: images-2.jpeg
[OCR Sonucu]:
10KUL BANANA CAM 400 GR I 11 SEYIDOGLU GÜL PEÇELI 700 11 JORSAN KASAP PEYNIR TAM YA 11
*69.59
*99.99
*339.90
-BEŞ YEDİTE UÇAK KEZİLİN K 11 VALI ÇEK LİĞİŞİM SİLER
*49.99
225.99
1 ADET X 4,99 TL
J.KER ÇOKEMEL POFTI DEV SA 11
*14.97
*85.99
SOKAĞA BİTKİNLİ BEKLİ İSLİ 5
HAY MARAS YOGURT 8KG
*34.90
(UMUTTA 30 LU+L)
*134.90
CUKURUZU SITAM CEK INDEGI 11
*49.99
1 ADET X 13.50 TL
*2

[Toplam Tutar Sorusu]:
Modelin Doğru Çıktısı: 1,423.28

📄 Dosya: images-3.jpeg
[OCR Sonucu]:
TARİH
06.01.2020 SAAT 15:59
#1340_0190
+9,50
ŞEKER TOZ 2 KG ÇAKAK TON BALIĞI ZY1EGGR TON BALIĞI ZY1EGGR TONSIZ KARIYE YEMİ *4,75
308
*9,95
308
308
308
308
308
308
SEKER KOP 1KG YENİ SEKER KOP 2KG YENİ KEÇMEK 500 GR
308
308
308
308
*5,45
CEVİZ EKMEK
308
308
308
308
*1,65
ÇEŞIT EKMEK
308
308
308
308
*1,45
SÜTLE 200 GR
308
308
RUS SALATA

[Toplam Tutar Sorusu]:
Modelin Doğru Çıktısı: 307.85

